# Preprocesamiento Textual

In [1]:
!pip install PyMuPDF pandas


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import fitz # PyMuPDF
import os
import pandas as pd
import re

### Toma de datos de los PDF

In [ ]:
#Ruta donde están los archivos pdf
root = r"Documentos_de_Inteligencia_artificial_GR 2\Apuntadores"

#lista para guardar resultados
documentos = []

#ciclo para recorrer los archivos pdf
for archivo in os.listdir(root):
    if archivo.lower().endswith(".pdf"):
        
        #creación de la ruta del archivo
        ruta_pdf = os.path.join(root, archivo)
        
        #abrir pdf con pymupdf
        with fitz.open(ruta_pdf) as pdf:
            texto_total = ""
            
            #recorrido por cada página para extraer texto
            #usar flags para mejor extracción de texto con Unicode
            for pagina in pdf:
                texto_total += pagina.get_text("text", flags=0) + "\n"
        
        #diccionario temporal para guardar datos de los pdf
        documentos.append({
            "nombre_archivo": archivo,
            "ruta": ruta_pdf,
            "texto": texto_total.strip()  #quit espacios en el texto
        })

#Guardar diccionario en un dataframe
df_docs = pd.DataFrame(documentos)

df_docs.head()


,nombre_archivo,ruta,texto
0,10_SEMANA_AI_20251007_1-222887296.pdf,Documentos_de_Inteligencia_artificial_GR 2\Apu...,Apuntes IA Clase 7/10\nGianmarco Oporta P´erez...
1,10_SEMANA_AI_20251007_1.pdf,Documentos_de_Inteligencia_artificial_GR 2\Apu...,Redes Neuronales Convolucionales y\nBackpropag...
2,10_SEMANA_AI_20251009_1.pdf,Documentos_de_Inteligencia_artificial_GR 2\Apu...,Apuntes de clase #2\nLuis Felipe Calderón Pére...
3,11_Semana_AI_20251014_1.pdf,Documentos_de_Inteligencia_artificial_GR 2\Apu...,Apuntes IA Clase 14/10/2025\nJuan Jim´enez Val...
4,11_Semana_AI_20251014_2.pdf,Documentos_de_Inteligencia_artificial_GR 2\Apu...,"Inteligencia Artificial\nApuntes Semana 11, Cl..."


### Extracción de metadatos

In [ ]:
#Expresion regular para extraer los datos del nombre del archivo
patron_nombre = r"(\d+)_semana_ai_(\d{4})(\d{2})(\d{2})_(\d+)\.pdf"

#lista para guardar datos finales
datos_finales = []

#filtro de autores conocidos
autores_conocidos = [
    "acuna lopez rodolfo david",
    "araya ortega fabian enrique",
    "benavides villegas luis fernando",
    "brenes martinez david",
    "brenes reyes fernando daniel",
    "brenes torres isaac david",
    "calderon perez luis felipe",
    "campos cerdas mauricio alonso",
    "carranza jimenez kevin josue",
    "diaz barboza fabian esteban",
    "espinoza aguilar dario",
    "gomez brenes gerardo alberto",
    "gonzalez sanchez luis alfredo",
    "jimenez salgado joselyn priscilla",
    "jimenez valverde juan diego",
    "murillo campos ian david",
    "naranjo masis alex steven",
    "oporta perez gianmarco",
    "quesada rodriguez jose pablo",
    "quesada sanchez mariana",
    "rodriguez camacho kendall andres",
    "rodriguez cano juan pablo",
    "rojas chacon sahid edgardo",
    "rojas obando nelson armando",
    "rojas rojas javier alonso",
    "sanchez araya brandon emmanuel",
    "sanchez rojas andres",
    "urena bermudez andrey",
    "varela venegas julio josue",
    "vargas solis rafael guillermo",
    "vasquez concepcion ashley lizeth",
    "vega suazo eder jose"
]

#segundo filtro de apellidos para mejorar la busqueda de autores
apellidos_autores = [
    "acuna", "araya", "benavides", "brenes", "calderon", "campos",
    "carranza", "diaz", "espinoza", "gomez", "gonzalez", "jimenez",
    "murillo", "naranjo", "oporta", "quesada", "rodriguez", "rojas",
    "sanchez", "urena", "varela", "vargas", "vasquez", "vega"
]

for i, fila in df_docs.iterrows():
    nombre = fila["nombre_archivo"]
    texto = fila["texto"]

    #extraer datos del nombre del archivo
    match = re.search(patron_nombre, nombre.lower())
    if match:
        semana = int(match.group(1))
        anio = int(match.group(2))
        dia = int(match.group(3))
        mes = int(match.group(4))
        apunte = int(match.group(5))
        fecha = f"{anio}-{mes:02d}-{dia:02d}"
    else:
        semana = None
        fecha = None
        apunte = None

    #buscar titulo y autor dentro de los pdf
    lineas = texto.splitlines()
    titulo = lineas[0].strip() if lineas else None

    autor = None
    email = None

    #recorrer el texto extraido del pdf linea por linea
    for linea in lineas[:15]:  
        linea_limpia = linea.strip().lower()
        
        #filtro para ubicar el correo electronico
        if "@estudiantec.cr" in linea_limpia:
            email = linea.strip()
            
        #buscar autor con coincidencias de los filtros
        if not autor and len(linea_limpia.split()) >= 2:
            # Verificar si contiene apellidos de estudiantes
            for apellido in apellidos_autores:
                if apellido in linea_limpia:
                    # Verificar que no contenga palabras del profesor
                    if "pacheco" not in linea_limpia and "portuguez" not in linea_limpia:
                        # Verificar que no sea el título ni otras líneas de metadata
                        if "abstract" not in linea_limpia and "index terms" not in linea_limpia:
                            autor = linea.strip()
                            break

   #Extraer Abstract
    abstract = None
    abstract_match = re.search(r"Abstract[—\-\s]+(.*?)(?:Index Terms|I\.|$)", texto, re.DOTALL | re.IGNORECASE)
    if abstract_match:
        abstract = abstract_match.group(1).strip()

    #Guardado de datos organizados
    datos_finales.append({
        "semana": semana,
        "fecha": fecha,
        "apunte": apunte,
        "titulo": titulo,
        "autor": autor,
        "email": email,
        "abstract": abstract,
        "texto": texto.strip(),
        "fuente": nombre
    })

# Crear un nuevo DataFrame con los metadatos enriquecidos
df_docs_full = pd.DataFrame(datos_finales)

# Mostrar una vista previa de los primeros registros
df_docs_full.head()


,semana,fecha,apunte,titulo,autor,email,abstract,texto,fuente
0,NaN,None,NaN,Apuntes IA Clase 7/10,Gianmarco Oporta P´erez,gooporta@estudiantec.cr,El presente documento recopila los apuntes de ...,Apuntes IA Clase 7/10\nGianmarco Oporta P´erez...,10_SEMANA_AI_20251007_1-222887296.pdf
1,10.0,2025-07-10,1.0,Redes Neuronales Convolucionales y,None,rodolfoide69@estudiantec.cr,En este documento podr´a encontrar informaci´o...,Redes Neuronales Convolucionales y\nBackpropag...,10_SEMANA_AI_20251007_1.pdf
2,10.0,2025-09-10,1.0,Apuntes de clase #2,None,None,None,Apuntes de clase #2\nLuis Felipe Calderón Pére...,10_SEMANA_AI_20251009_1.pdf
3,11.0,2025-14-10,1.0,Apuntes IA Clase 14/10/2025,"como los filtros, campos receptivos, stride, p...",juand0908@estudiantec.cr,Este documento resume los conceptos clave vist...,Apuntes IA Clase 14/10/2025\nJuan Jim´enez Val...,11_Semana_AI_20251014_1.pdf
4,11.0,2025-14-10,2.0,Inteligencia Artificial,Luis Fernando Benavides Villegas,lubenavides@estudiantec.cr,Este documento recopila los apuntes de la clas...,"Inteligencia Artificial\nApuntes Semana 11, Cl...",11_Semana_AI_20251014_2.pdf


### Limpieza de datos

In [ ]:
import unicodedata

def eliminar_tildes(texto):
    """
    Elimina tildes y acentos manejando:
    1. Unicode compuesto (é)
    2. Unicode descompuesto (e + ´)
    3. Caracteres literales de acento (´, `, ¨, ^, ~)
    4. Caracteres especiales resultantes (ı, ȷ, etc.)
    """
    if not texto or not isinstance(texto, str):
        return texto
    
    # Paso 1: Eliminar caracteres literales de acento que aparecen solos
    acentos_literales = ['´', '`', '¨', '^', '~', '¯', '˜', '¸', 'ˆ', '˙', '˚', '˝']
    for acento in acentos_literales:
        texto = texto.replace(acento, '')
    
    #Paso 2: Normalizar a NFD (descomponer caracteres acentuados)
    texto_nfd = unicodedata.normalize('NFD', texto)
    
    # Paso 3: Filtrar marcas diacríticas (categoría Mn)
    texto_sin_tildes = ''.join(
        char for char in texto_nfd 
        if unicodedata.category(char) != 'Mn'
    )
    
    # Paso 4: Normalizar a NFC (recomponer)
    texto_final = unicodedata.normalize('NFC', texto_sin_tildes)
    
    # Paso 5: Reemplazar caracteres especiales que quedan
    # ı (dotless i) -> i
    # ȷ (dotless j) -> j
    # ø (o con barra) -> o
    # etc.
    reemplazos_especiales = {
        'ı': 'i',  # Latin small letter dotless i
        'ȷ': 'j',  # Latin small letter dotless j
        'ø': 'o',  # Latin small letter o with stroke
        'Ø': 'O',  # Latin capital letter O with stroke
        'ł': 'l',  # Latin small letter l with stroke
        'Ł': 'L',  # Latin capital letter L with stroke
        'ð': 'd',  # Latin small letter eth
        'Ð': 'D',  # Latin capital letter ETH
        'þ': 'th', # Latin small letter thorn
        'Þ': 'Th', # Latin capital letter THORN
        'ß': 'ss', # Latin small letter sharp s
    }
    
    for especial, normal in reemplazos_especiales.items():
        texto_final = texto_final.replace(especial, normal)
    
    return texto_final

def limpiar_campo_simple(texto):
    """Limpia campos individuales (autor, email, título)"""
    if not texto or not isinstance(texto, str):
        return texto
    
    # 1. Eliminar tildes
    texto = eliminar_tildes(texto)
    
    # 2. Convertir a minúsculas
    texto = texto.lower()
    
    # 3. Normalizar espacios
    texto = " ".join(texto.split())
    
    return texto

def limpiar_texto(texto):
    if not texto or not isinstance(texto, str):
        return texto
    
    # 1. Eliminar tildes
    texto = eliminar_tildes(texto)
    
    # 2. Convertir a minúsculas
    texto = texto.lower()
    
    # 3. Normalizar espacios y saltos de línea
    texto = " ".join(texto.split())
    
    return texto

def validar_autor(autor):
    if not autor or not isinstance(autor, str):
        return autor
    
    autor_limpio = limpiar_campo_simple(autor)
    
    # Verificar si está en la lista de autores conocidos
    for nombre_conocido in autores_conocidos:
        if nombre_conocido in autor_limpio or autor_limpio in nombre_conocido:
            return autor
    
    return autor  # Si no se encuentra, mantener el original

# Aplicar limpieza a todos los campos del DataFrame
df_docs_full["titulo"] = df_docs_full["titulo"].apply(limpiar_campo_simple)
df_docs_full["autor"] = df_docs_full["autor"].apply(lambda x: validar_autor(x)).apply(limpiar_campo_simple)
df_docs_full["email"] = df_docs_full["email"].apply(limpiar_campo_simple)
df_docs_full["abstract"] = df_docs_full["abstract"].apply(limpiar_texto)
df_docs_full["texto"] = df_docs_full["texto"].apply(limpiar_texto)
df_docs_full["fuente"] = df_docs_full["fuente"].apply(limpiar_campo_simple)

# Mostrar todas las columnas (incluyendo texto limpio)
df_docs_full.head()


,semana,fecha,apunte,titulo,autor,email,abstract,texto,fuente
0,NaN,None,NaN,apuntes ia clase 7/10,gianmarco oporta perez,gooporta@estudiantec.cr,el presente documento recopila los apuntes de ...,apuntes ia clase 7/10 gianmarco oporta perez i...,10_semana_ai_20251007_1-222887296.pdf
1,10.0,2025-07-10,1.0,redes neuronales convolucionales y,None,rodolfoide69@estudiantec.cr,en este documento podra encontrar informacion ...,redes neuronales convolucionales y backpropaga...,10_semana_ai_20251007_1.pdf
2,10.0,2025-09-10,1.0,apuntes de clase #2,None,None,None,apuntes de clase #2 luis felipe calderon perez...,10_semana_ai_20251009_1.pdf
3,11.0,2025-14-10,1.0,apuntes ia clase 14/10/2025,"como los filtros, campos receptivos, stride, p...",juand0908@estudiantec.cr,este documento resume los conceptos clave vist...,apuntes ia clase 14/10/2025 juan jimenez valve...,11_semana_ai_20251014_1.pdf
4,11.0,2025-14-10,2.0,inteligencia artificial,luis fernando benavides villegas,lubenavides@estudiantec.cr,este documento recopila los apuntes de la clas...,"inteligencia artificial apuntes semana 11, cla...",11_semana_ai_20251014_2.pdf


### Guardar DataFrame procesado

### congelar dataframe en un .parquet

In [6]:
!pip install -U pandas pyarrow fastparquet


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
#Guardar el DataFrame limpio en formato Parquet
# Esto permite versionarlo y cargarlo rápidamente después
output_path = "df_docs_procesado.parquet"

df_docs_full.to_parquet(output_path, engine='pyarrow', compression='snappy')
print(f"DataFrame guardado exitosamente en: {output_path}")
print(f"Total de documentos: {len(df_docs_full)}")
print(f"Columnas guardadas: {list(df_docs_full.columns)}")

DataFrame guardado exitosamente en: df_docs_procesado.parquet
Total de documentos: 46
Columnas guardadas: ['semana', 'fecha', 'apunte', 'titulo', 'autor', 'email', 'abstract', 'texto', 'fuente']


In [6]:
import os
import pandas as pd

def guardar_parquet_con_fallback(df, ruta):
    try:
        import pyarrow as pa  
        try:
            pa.unregister_extension_type("pandas.period")
        except Exception:
            pass
        df.to_parquet(ruta, index=False, engine="pyarrow", compression="snappy")
        print(f"Guardado con pyarrow en: {ruta}")
    except Exception as e:
        print("pyarrow falló:", e, "\n→ Intentando con fastparquet…")
        df.to_parquet(ruta, index=False, engine="fastparquet", compression="snappy")
        print(f"Guardado con fastparquet en: {ruta}")

carpeta_salida = "data"
os.makedirs(carpeta_salida, exist_ok=True)
ruta_parquet = os.path.join(carpeta_salida, "apuntes_clean_v1.parquet")

guardar_parquet_con_fallback(df_docs_full, ruta_parquet)

Guardado con pyarrow en: data\apuntes_clean_v1.parquet


### Tecnicas de segmentación

#### Fixed-size Chunking with sliding window

In [ ]:
#Fixed-size chunking con sliding window
#Entrada: dataset limpio cargado desde data/*.parquet (sin modificar el DF original)
#Salida: df_chunks_sliding con un chunk por fila, preservando metadatos útiles
#Regla: stride = chunk_size - overlap

import os
import pandas as pd
from typing import List, Dict

# Lee el dataset "congelado" desde /data

def load_dataset():
    candidatos = [
        os.path.join("data", "apuntes_clean_v1.parquet"),
        os.path.join("data", "df_docs_procesado.parquet"),
    ]
    for ruta in candidatos:
        if os.path.exists(ruta):
            # intentar pyarrow, luego fastparquet
            try:
                return pd.read_parquet(ruta, engine="pyarrow")
            except Exception:
                try:
                    return pd.read_parquet(ruta, engine="fastparquet")
                except Exception:
                    pass
    raise FileNotFoundError(
        "No se encontró el dataset en data/*.parquet"
    )

# Función de chunking con ventana deslizante
# Retorna una lista de dicts con: start, end, length, chunk, chunk_index.
def chunk_text_sliding_window(texto: str, chunk_size: int = 1000, overlap: int = 200) -> List[Dict]:

    if not isinstance(texto, str) or not texto:
        return []

    if chunk_size <= 0:
        raise ValueError("chunk_size debe ser > 0")
    if overlap < 0 or overlap >= chunk_size:
        raise ValueError("overlap debe estar en [0, chunk_size)")

    stride = chunk_size - overlap
    chunks: List[Dict] = []

    n = len(texto)
    idx = 0
    start = 0
    while start < n:
        end = min(start + chunk_size, n)
        fragmento = texto[start:end]
        if fragmento.strip():
            chunks.append({
                "chunk_index": idx,
                "start": start,
                "end": end,
                "length": end - start,
                "chunk": fragmento,
            })
            idx += 1
        if end == n:
            break
        start += stride

    return chunks

# Cargar dataset desde /data sin tocar el DF original
pd.set_option('display.max_colwidth', None)
df_src = load_dataset()  # columnas esperadas: semana, fecha, apunte, titulo, autor, email, abstract, texto, fuente

# Aplicar segmentación a todo el dataset y construir un DataFrame de chunks
registros = []
for row in df_src.itertuples(index=False):
    texto_doc = getattr(row, "texto")
    fuente = getattr(row, "fuente")
    for ch in chunk_text_sliding_window(texto_doc, chunk_size=1000, overlap=200):
        registros.append({
            "fuente": fuente,
            "semana": getattr(row, "semana", None),
            "fecha": getattr(row, "fecha", None),
            "apunte": getattr(row, "apunte", None),
            "titulo": getattr(row, "titulo", None),
            "autor": getattr(row, "autor", None),
            "chunk_index": ch["chunk_index"],
            "start": ch["start"],
            "end": ch["end"],
            "length": ch["length"],
            "chunk": ch["chunk"],
        })

df_chunks_sliding = pd.DataFrame(registros).sort_values(["fuente", "chunk_index"]).reset_index(drop=True)

# Resumen y vista
num_docs = df_src.shape[0]
num_chunks = df_chunks_sliding.shape[0]
print(f" Documentos: {num_docs} | Chunks generados: {num_chunks} | Promedio por doc: {num_chunks / max(num_docs,1):.2f}")

df_chunks_sliding.head(10)

# Guardar a disco para versionar resultados de segmentación
try:
    carpeta_salida = "data"
    os.makedirs(carpeta_salida, exist_ok=True)
    ruta_chunks = os.path.join(carpeta_salida, "chunks_sliding_v1.parquet")
    # Reusar el helper si existe en el notebook, sino guardar con parquet simple
    try:
        guardar_parquet_con_fallback(df_chunks_sliding, ruta_chunks)
    except NameError:
        # Si no está definida la función auxiliar, intentamos parquet directo con fallback simple
        try:
            df_chunks_sliding.to_parquet(ruta_chunks, engine="pyarrow", index=False, compression="snappy")
            print(f"Chunks guardados en: {ruta_chunks}")
        except Exception:
            raise
except Exception as e:
    print("No se guardó parquet de chunks (opcional):", e)

 Documentos: 46 | Chunks generados: 620 | Promedio por doc: 13.48
Guardado con pyarrow en: data\chunks_sliding_v1.parquet


#### Recursive chunking 

Este método intenta mantener límites “naturales” del texto antes de forzarlo a tamaños fijos:
- Empieza dividiendo por separadores más fuertes a más débiles (párrafos → líneas → oraciones → signos → espacios).
- Si un fragmento sigue siendo largo, vuelve a dividirlo recursivamente con el siguiente separador.
- Al final, recombina piezas en chunks de tamaño objetivo con un solapamiento (overlap) definido para preservar contexto entre chunks.

Ventajas:
- Chunks más “semánticos” (menos cortes bruscos a mitad de ideas).
- Control fino del tamaño y del solapamiento.

Parámetros típicos:
- chunk_size: 1000 caracteres
- overlap: 200 caracteres
- separadores: ["\n\n", "\n", ". ", "; ", ", ", " "] (de fuerte → débil)

In [8]:
# Segmentación recursiva desde data/apuntes_clean_v1.parquet y guardado en data/chunks_recursive_v1.parquet
import os
import re
import pandas as pd
from typing import List

# Reutilizamos el loader robusto (pyarrow -> fastparquet)
def load_dataset():
    rutas_intentos = [
        os.path.join('data', 'apuntes_clean_v1.parquet'),
        'apuntes_clean_v1.parquet',
    ]
    ultimo_error = None
    for ruta in rutas_intentos:
        if os.path.exists(ruta):
            try:
                try:
                    return pd.read_parquet(ruta, engine='pyarrow')
                except Exception as e1:
                    print(f"pyarrow falló: {e1}\n→ Intentando con fastparquet…")
                    try:
                        return pd.read_parquet(ruta, engine='fastparquet')
                    except Exception as e2:
                        ultimo_error = (e1, e2)
                        print(f"fastparquet también falló: {e2}")
            except Exception as e:
                ultimo_error = e
    raise RuntimeError(f"No se pudo cargar el dataset .parquet. Último error: {ultimo_error}")


# Utilidades: normalizar espacios y asegurar strings
_ws_re = re.compile(r"\s+")


def _s(text):
    if text is None:
        return ""
    return str(text)

# Split recursivo por una lista jerárquica de separadores
# separadores: de fuerte → débil (p. ej. párrafos → líneas → oraciones → espacios)

def recursive_split(text: str, separators: List[str], max_len: int) -> List[str]:
    text = _s(text).strip()
    if not text:
        return []
    # Caso base: si ya es corto, devolver tal cual
    if len(text) <= max_len:
        return [text]

    if not separators:
        # Sin separadores disponibles: forzar corte duro
        return [text[i:i+max_len] for i in range(0, len(text), max_len)]

    sep = separators[0]
    rest = separators[1:]

    # Dividir por el separador actual; si el separador es espacio simple, usar split(' ')
    if sep == ' ':
        parts = text.split(' ')
        glue = ' '
    else:
        parts = text.split(sep)
        glue = sep

    # Si el separador no generó cortes (texto sin ese separador), avanzar al siguiente
    if len(parts) == 1:
        return recursive_split(text, rest, max_len)

    # Repartir recursivamente cada parte si excede tamaño
    chunks = []
    current = []
    current_len = 0

    def flush_current():
        nonlocal current, current_len
        if current:
            combined = glue.join(current).strip()
            if combined:
                if len(combined) <= max_len:
                    chunks.append(combined)
                else:
                    # Aún grande: bajar a siguiente separador
                    chunks.extend(recursive_split(combined, rest, max_len))
        current = []
        current_len = 0

    for p in parts:
        piece = p.strip()
        if not piece:
            # Conserva separador donde sea útil; evitamos cadenas vacías consecutivas
            if current and glue:
                # Empuja un separador lógico en la recombinación
                current.append('')
            continue
        prospective_len = (current_len + (len(glue) if current else 0) + len(piece))
        if prospective_len <= max_len:
            # Aún cabe en el paquete actual
            if current:
                current.append(piece)
                current_len = prospective_len
            else:
                current = [piece]
                current_len = len(piece)
        else:
            # Cierra paquete actual y decide sobre piece
            flush_current()
            if len(piece) <= max_len:
                current = [piece]
                current_len = len(piece)
            else:
                # La pieza sola ya excede: desciende de nivel
                chunks.extend(recursive_split(piece, rest, max_len))
                current = []
                current_len = 0

    flush_current()
    return chunks

# Recombina listas de trozos cortos en ventanas con solapamiento, manteniendo tamaño objetivo

def pack_with_overlap(frags: List[str], chunk_size: int, overlap: int) -> List[str]:
    if not frags:
        return []
    # Normaliza y filtra vacíos
    norm = []
    for f in frags:
        s = _ws_re.sub(' ', _s(f)).strip()
        if s:
            norm.append(s)
    if not norm:
        return []

    chunks = []
    buf = []
    buf_len = 0

    def emit():
        nonlocal buf, buf_len
        if buf:
            out = ' '.join(buf).strip()
            if out:
                chunks.append(out)
        buf = []
        buf_len = 0

    for frag in norm:
        if buf_len == 0:
            buf.append(frag)
            buf_len = len(frag)
            continue
        prospective = buf_len + 1 + len(frag)  # +1 por espacio
        if prospective <= chunk_size:
            buf.append(frag)
            buf_len = prospective
        else:
            # Emite actual y aplica solapamiento aproximado por palabras
            emit()
            if overlap > 0 and chunks:
                tail = chunks[-1]
                # Toma ~overlap caracteres desde el final, cortado por palabras
                # Luego extrae últimas palabras para reconstruir contexto
                if len(tail) > overlap:
                    tail_ctx = tail[-overlap:]
                    # Evita iniciar en medio de palabra
                    tail_ctx = tail_ctx[tail_ctx.find(' ')+1:] if ' ' in tail_ctx else tail_ctx
                    if tail_ctx:
                        buf = [tail_ctx]
                        buf_len = len(tail_ctx)
                    else:
                        buf = []
                        buf_len = 0
                else:
                    buf = [tail]
                    buf_len = len(tail)
            else:
                buf = []
                buf_len = 0
            # Añade el fragmento actual (puede iniciar nuevo buffer)
            if buf_len == 0:
                buf = [frag]
                buf_len = len(frag)
            else:
                prospective = buf_len + 1 + len(frag)
                if prospective <= chunk_size:
                    buf.append(frag)
                    buf_len = prospective
                else:
                    emit()
                    buf = [frag]
                    buf_len = len(frag)

    emit()
    return chunks

# Pipeline: carga → split recursivo → empaquetado con solapamiento → DataFrame y guardado

df_docs_frozen = load_dataset()
assert 'texto' in df_docs_frozen.columns, "El dataset cargado debe contener la columna 'texto'"

separators = ["\n\n", "\n", ". ", "; ", ", ", " "]
chunk_size = 1000
overlap = 200

registros = []
for idx, row in df_docs_frozen.iterrows():
    texto = _s(row.get('texto', ''))
    if not texto.strip():
        continue
    # 1) Split recursivo por jerarquía de separadores
    frags = recursive_split(texto, separators, max_len=chunk_size)
    # 2) Empaquetar con solapamiento para uniformar tamaño objetivo
    chunks = pack_with_overlap(frags, chunk_size=chunk_size, overlap=overlap)

    for c_idx, c in enumerate(chunks):
        registros.append({
            'fuente': row.get('fuente'),
            'semana': row.get('semana'),
            'fecha': row.get('fecha'),
            'apunte': row.get('apunte'),
            'titulo': row.get('titulo'),
            'autor': row.get('autor'),
            'chunk_id': f"rec_{idx}_{c_idx}",
            'chunk_text': c,
            'chunk_len': len(c),
        })

# DataFrame de salida
chunks_recursive_df = pd.DataFrame(registros)
print(f" Documentos: {len(df_docs_frozen)} | Chunks generados (rec): {len(chunks_recursive_df)} | Promedio por doc: {len(chunks_recursive_df)/max(1,len(df_docs_frozen)):.2f}")
print(chunks_recursive_df['chunk_len'].describe().round(1))

# Guardado robusto en data/chunks_recursive_v1.parquet 
os.makedirs('data', exist_ok=True)
salida = os.path.join('data', 'chunks_recursive_v1.parquet')
try:
    chunks_recursive_df.to_parquet(salida, engine='pyarrow', index=False)
    print(f"Guardado con pyarrow en: {salida}")
except Exception as e1:
    print(f"pyarrow falló: {e1}\n→ Intentando con fastparquet…")
    try:
        chunks_recursive_df.to_parquet(salida, engine='fastparquet', index=False)
        print(f"Guardado con fastparquet en: {salida}")
    except Exception as e2:
        print(f"fastparquet también falló: {e2}")
        raise

 Documentos: 46 | Chunks generados (rec): 988 | Promedio por doc: 21.48
count     988.0
mean      595.1
std       361.9
min       180.0
25%       196.0
50%       836.5
75%       943.2
max      1000.0
Name: chunk_len, dtype: float64
Guardado con pyarrow en: data\chunks_recursive_v1.parquet


# Tokenización y Embeddings

In [11]:
!pip install -U pandas pyarrow numpy tqdm tenacity
!pip install -U openai python-dotenv


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


### Instalación de FAISS para Base de Datos Vectorial

FAISS (Facebook AI Similarity Search) es una biblioteca optimizada para búsqueda de similitud en espacios vectoriales de alta dimensión.

**Ventajas**:
- Búsqueda extremadamente rápida (millones de vectores/segundo)
- Bajo uso de memoria
- Soporte para CPU y GPU
- Ideal para prototipado y producción local

**Uso en este proyecto**:
- Crearemos 2 índices independientes (sliding y recursive)
- Cada índice almacenará ~600-1000 vectores de 1536 dimensiones
- Métrica: producto interno (equivalente a cosine similarity con vectores normalizados)

In [12]:
# Instalar FAISS (versión CPU) y tiktoken para tokenización de OpenAI
!pip install -q faiss-cpu tiktoken


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


### Configuración del Cliente OpenAI

Para generar embeddings con `text-embedding-3-small`, necesitamos:
1. Una API key válida de OpenAI
2. El cliente de la librería openai (v1.0+)
3. Un archivo `.env` con la configuración (más seguro)

**Configuración con archivo .env** (recomendado):
1. Edita el archivo `.env` en la raíz del proyecto
2. Reemplaza `sk-proj-tu-api-key-aqui` con tu API key real
3. El archivo `.env` ya está en `.gitignore` para proteger tus credenciales

**Formato del archivo .env**:
```
OPENAI_API_KEY=sk-proj-tu-api-key-real-aqui
```

La librería `python-dotenv` cargará automáticamente estas variables.

In [9]:
import os
from openai import OpenAI
from dotenv import load_dotenv

# Cargar variables de entorno desde el archivo .env
load_dotenv()

# Verificar que la API key esté cargada
api_key = os.getenv("OPENAI_API_KEY")

# Inicializar cliente OpenAI
try:
    client = OpenAI(api_key=api_key)
    print("Cliente OpenAI inicializado correctamente")
    print(f"Modelo a usar: text-embedding-3-small (1536 dimensiones)")
    print(f"API key cargada desde .env: {api_key[:20]}...{api_key[-4:]}")
except Exception as e:
    print(f"Error al inicializar cliente OpenAI: {e}")
    raise

Cliente OpenAI inicializado correctamente
Modelo a usar: text-embedding-3-small (1536 dimensiones)
API key cargada desde .env: sk-proj-BSJOpTOHcMSe...rc0A


In [10]:
import numpy as np
from typing import List
from tqdm import tqdm
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type
from openai import OpenAI, RateLimitError, APITimeoutError, APIConnectionError

@retry(
    stop=stop_after_attempt(5),
    wait=wait_exponential(multiplier=1, min=2, max=60),
    retry=retry_if_exception_type((RateLimitError, APITimeoutError, APIConnectionError)),
    reraise=True
)
# Llamada a la API de embeddings con retry automático
def _call_embedding_api(client: OpenAI, texts: List[str], model: str) -> List[List[float]]:
    response = client.embeddings.create(
        input=texts,
        model=model
    )
    return [item.embedding for item in response.data]


def generate_embeddings_batch(
    texts: List[str],
    client: OpenAI,
    model: str = "text-embedding-3-small",
    batch_size: int = 100,
    normalize: bool = True,
    show_progress: bool = True
) -> np.ndarray:
    
    if not texts:
        return np.array([])
    
    # Filtrar textos vacíos y mantener índices originales
    valid_indices = [i for i, t in enumerate(texts) if t and str(t).strip()]
    valid_texts = [texts[i] for i in valid_indices]
    
    if not valid_texts:
        print("Advertencia: todos los textos están vacíos")
        return np.array([])
    
    print(f"Generando embeddings para {len(valid_texts)} textos...")
    print(f"   Modelo: {model}")
    print(f"   Batch size: {batch_size}")
    print(f"   Normalización: {'Sí' if normalize else 'No'}")
    
    all_embeddings = []
    
    # Procesar en lotes con progress bar
    iterator = range(0, len(valid_texts), batch_size)
    if show_progress:
        iterator = tqdm(iterator, desc="Generando embeddings", unit="batch")
    
    for i in iterator:
        batch = valid_texts[i:i + batch_size]
        
        try:
            # Llamar API con retry automático
            batch_embeddings = _call_embedding_api(client, batch, model)
            all_embeddings.extend(batch_embeddings)
            
        except Exception as e:
            print(f"\nError en batch {i//batch_size + 1}: {e}")
            raise
    
    # Convertir a numpy array
    embeddings_array = np.array(all_embeddings, dtype=np.float32)
    
    # Normalizar si se solicita (requerido para cosine similarity con FAISS)
    if normalize:
        norms = np.linalg.norm(embeddings_array, axis=1, keepdims=True)
        embeddings_array = embeddings_array / norms
    
    print(f"Embeddings generados: shape {embeddings_array.shape}")
    print(f"   Rango de valores: [{embeddings_array.min():.4f}, {embeddings_array.max():.4f}]")
    
    # Si había textos vacíos, crear array completo con zeros en posiciones vacías
    if len(valid_indices) < len(texts):
        full_embeddings = np.zeros((len(texts), embeddings_array.shape[1]), dtype=np.float32)
        full_embeddings[valid_indices] = embeddings_array
        return full_embeddings
    
    return embeddings_array


# Test rápido de la función (comentar después de verificar)
#print("\nTest de la función de embeddings:")
#test_texts = ["inteligencia artificial", "aprendizaje automatico", "redes neuronales"]
#test_embeddings = generate_embeddings_batch(test_texts, client, batch_size=3)
#print(f"Test exitoso: {len(test_texts)} textos → shape {test_embeddings.shape}")

### Función para Generar Embeddings en Lotes

Esta función implementa las mejores prácticas para generar embeddings con la API de OpenAI:

**Características**:
- **Procesamiento en batch**: Procesa múltiples textos por request (hasta 100 por lote según límites de OpenAI)
- **Retry automático**: Usa `tenacity` para reintentar en caso de errores temporales (rate limits, timeouts)
- **Progress bar**: Muestra el progreso con `tqdm` para visualizar el avance
- **Normalización**: Los vectores se normalizan para usar cosine similarity con FAISS
- **Manejo de errores**: Captura y reporta errores específicos

**Parámetros**:
- `texts`: Lista de strings a convertir en embeddings
- `model`: Modelo de OpenAI (por defecto "text-embedding-3-small")
- `batch_size`: Cantidad de textos por request (máx 100)
- `normalize`: Si normalizar vectores para cosine similarity (recomendado para FAISS)

In [12]:
import faiss
import json
import os
import pandas as pd

# cargar chunks desde data/chunks_sliding_v1.parquet
print("=" * 80)
print("PROCESANDO CHUNKS SLIDING WINDOW")
print("=" * 80)

ruta_chunks_sliding = os.path.join("data", "chunks_sliding_v1.parquet")

try:
    df_sliding = pd.read_parquet(ruta_chunks_sliding, engine="fastparquet")
    print(f"Chunks cargados: {len(df_sliding)} registros")
    print(f"   Columnas: {list(df_sliding.columns)}")
except Exception as e:
    print(f" Error al cargar chunks: {e}")
    raise

# Extraer textos de la columna 'chunk'
texts_sliding = df_sliding["chunk"].tolist()
print(f" Textos a procesar: {len(texts_sliding)}")

#Generar embeddings para los chunks con ventana deslizante
print("\n" + "=" * 80)
print("GENERANDO EMBEDDINGS")
print("=" * 80)

embeddings_sliding = generate_embeddings_batch(
    texts=texts_sliding,
    client=client,
    model="text-embedding-3-small",
    batch_size=100,
    normalize=True,
    show_progress=True
)

print(f"\n Embeddings generados: {embeddings_sliding.shape}")

# Crear índice FAISS para los embeddings generados
print("\n" + "=" * 80)
print("CREANDO ÍNDICE FAISS")
print("=" * 80)

# Dimensión de los embeddings (1536 para text-embedding-3-small)
dimension = embeddings_sliding.shape[1]

# Crear índice FAISS con producto interno (equivale a cosine similarity con vectores normalizados)
index_sliding = faiss.IndexFlatIP(dimension)

# Agregar vectores al índice
index_sliding.add(embeddings_sliding)

print(f" Índice FAISS creado")
print(f"   Tipo: IndexFlatIP (Inner Product / Cosine Similarity)")
print(f"   Dimensión: {index_sliding.d}")
print(f"   Vectores almacenados: {index_sliding.ntotal}")

# Guardar índice y metadata
print("\n" + "=" * 80)
print("GUARDANDO ÍNDICE Y METADATA")
print("=" * 80)

# Crear directorio vectordb
os.makedirs("vectordb", exist_ok=True)

# Guardar índice FAISS
index_path = os.path.join("vectordb", "faiss_index_sliding.bin")
faiss.write_index(index_sliding, index_path)
print(f" Índice guardado: {index_path}")

# Guardar metadata (información de los chunks para recuperación)
metadata_sliding = {
    "index_type": "IndexFlatIP",
    "dimension": dimension,
    "total_vectors": int(index_sliding.ntotal),
    "model": "text-embedding-3-small",
    "chunking_method": "fixed-size sliding window",
    "chunk_size": 1000,
    "overlap": 200,
    "normalized": True,
    "columns": list(df_sliding.columns),
    "created_at": pd.Timestamp.now().isoformat()
}

metadata_path = os.path.join("vectordb", "faiss_index_sliding_metadata.json")
with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(metadata_sliding, f, indent=2, ensure_ascii=False)
print(f" Metadata guardada: {metadata_path}")

# Guardar backup con embeddings (solo parquet, sin pickle)
print("\n" + "=" * 80)
print("GUARDANDO BACKUP CON EMBEDDINGS")
print("=" * 80)

# Agregar embeddings al DataFrame como nueva columna
df_sliding_with_embeddings = df_sliding.copy()
df_sliding_with_embeddings["embedding"] = list(embeddings_sliding)

backup_path = os.path.join("data", "embeddings_sliding_v1.parquet")
df_sliding_with_embeddings.to_parquet(backup_path, engine="pyarrow", index=False)
print(f" Backup guardado: {backup_path}")

# Resumen final
print("\n" + "=" * 80)
print(" PROCESAMIENTO COMPLETADO: CHUNKS SLIDING WINDOW")
print("=" * 80)
print(f" Chunks procesados: {len(df_sliding)}")
print(f" Embeddings generados: {embeddings_sliding.shape}")
print(f"  Índice FAISS: {index_sliding.ntotal} vectores")
print(f"\n Archivos generados:")
print(f"   - {index_path}")
print(f"   - {metadata_path}")
print(f"   - {backup_path}")
print("=" * 80)

PROCESANDO CHUNKS SLIDING WINDOW
Chunks cargados: 620 registros
   Columnas: ['fuente', 'semana', 'fecha', 'apunte', 'titulo', 'autor', 'chunk_index', 'start', 'end', 'length', 'chunk']
 Textos a procesar: 620

GENERANDO EMBEDDINGS
Generando embeddings para 620 textos...
   Modelo: text-embedding-3-small
   Batch size: 100
   Normalización: Sí


Generando embeddings:   0%|          | 0/7 [00:00<?, ?batch/s]

Generando embeddings: 100%|██████████| 7/7 [00:09<00:00,  1.38s/batch]

Embeddings generados: shape (620, 1536)
   Rango de valores: [-0.1570, 0.1853]

 Embeddings generados: (620, 1536)

CREANDO ÍNDICE FAISS
 Índice FAISS creado
   Tipo: IndexFlatIP (Inner Product / Cosine Similarity)
   Dimensión: 1536
   Vectores almacenados: 620

GUARDANDO ÍNDICE Y METADATA
 Índice guardado: vectordb\faiss_index_sliding.bin
 Metadata guardada: vectordb\faiss_index_sliding_metadata.json

GUARDANDO BACKUP CON EMBEDDINGS
 Backup guardado: data\embeddings_sliding_v1.parquet

 PROCESAMIENTO COMPLETADO: CHUNKS SLIDING WINDOW
 Chunks procesados: 620
 Embeddings generados: (620, 1536)
  Índice FAISS: 620 vectores

 Archivos generados:
   - vectordb\faiss_index_sliding.bin
   - vectordb\faiss_index_sliding_metadata.json
   - data\embeddings_sliding_v1.parquet


### Procesamiento de Chunks Sliding Window

Ahora procesaremos los chunks de sliding window (620 chunks) para:
1. **Cargar** el archivo `data/chunks_sliding_v1.parquet`
2. **Generar embeddings** para cada chunk usando OpenAI
3. **Crear índice FAISS** (IndexFlatIP para cosine similarity)
4. **Guardar resultados**:
   - Índice FAISS: `vectordb/faiss_index_sliding.bin`
   - Metadata JSON: `vectordb/faiss_index_sliding_metadata.json`
   - Backup con embeddings: `data/embeddings_sliding_v1.parquet`


In [13]:
import faiss
import json
import os
import pandas as pd

# cargar chunks desde data/chunks_recursive_v1.parquet
print("=" * 80)
print("PROCESANDO CHUNKS RECURSIVE")
print("=" * 80)

ruta_chunks_recursive = os.path.join("data", "chunks_recursive_v1.parquet")

try:
    df_recursive = pd.read_parquet(ruta_chunks_recursive, engine="fastparquet")
    print(f" Chunks cargados: {len(df_recursive)} registros")
    print(f"   Columnas: {list(df_recursive.columns)}")
except Exception as e:
    print(f" Error al cargar chunks: {e}")
    raise

# Extraer textos de la columna 'chunk_text' (nota: diferente nombre que sliding)
texts_recursive = df_recursive["chunk_text"].tolist()
print(f" Textos a procesar: {len(texts_recursive)}")

# Generar embeddings para los chunks recursivos
print("\n" + "=" * 80)
print("GENERANDO EMBEDDINGS")
print("=" * 80)

embeddings_recursive = generate_embeddings_batch(
    texts=texts_recursive,
    client=client,
    model="text-embedding-3-small",
    batch_size=100,
    normalize=True,
    show_progress=True
)

print(f"\n Embeddings generados: {embeddings_recursive.shape}")

# Crear índice FAISS para los embeddings generados
print("\n" + "=" * 80)
print("CREANDO ÍNDICE FAISS")
print("=" * 80)

# Dimensión de los embeddings (1536 para text-embedding-3-small)
dimension = embeddings_recursive.shape[1]

# Crear índice FAISS con producto interno
index_recursive = faiss.IndexFlatIP(dimension)

# Agregar vectores al índice
index_recursive.add(embeddings_recursive)

print(f" Índice FAISS creado")
print(f"   Tipo: IndexFlatIP (Inner Product / Cosine Similarity)")
print(f"   Dimensión: {index_recursive.d}")
print(f"   Vectores almacenados: {index_recursive.ntotal}")

# Guardar índice y metadata
print("\n" + "=" * 80)
print("GUARDANDO ÍNDICE Y METADATA")
print("=" * 80)

# Crear directorio vectordb (ya existe, pero por si acaso)
os.makedirs("vectordb", exist_ok=True)

# Guardar índice FAISS
index_path = os.path.join("vectordb", "faiss_index_recursive.bin")
faiss.write_index(index_recursive, index_path)
print(f" Índice guardado: {index_path}")

# Guardar metadata
metadata_recursive = {
    "index_type": "IndexFlatIP",
    "dimension": dimension,
    "total_vectors": int(index_recursive.ntotal),
    "model": "text-embedding-3-small",
    "chunking_method": "recursive text splitter",
    "chunk_size": 1000,
    "overlap": 200,
    "separators": ["\n\n", "\n", ". ", "; ", ", ", " "],
    "normalized": True,
    "columns": list(df_recursive.columns),
    "created_at": pd.Timestamp.now().isoformat()
}

metadata_path = os.path.join("vectordb", "faiss_index_recursive_metadata.json")
with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(metadata_recursive, f, indent=2, ensure_ascii=False)
print(f" Metadata guardada: {metadata_path}")

# Guardar backup con embeddings (parquet)
print("\n" + "=" * 80)
print("GUARDANDO BACKUP CON EMBEDDINGS")
print("=" * 80)

# Agregar embeddings al DataFrame como nueva columna
df_recursive_with_embeddings = df_recursive.copy()
df_recursive_with_embeddings["embedding"] = list(embeddings_recursive)

backup_path = os.path.join("data", "embeddings_recursive_v1.parquet")
df_recursive_with_embeddings.to_parquet(backup_path, engine="pyarrow", index=False)
print(f" Backup guardado: {backup_path}")

# Resumen final
print("\n" + "=" * 80)
print(" PROCESAMIENTO COMPLETADO: CHUNKS RECURSIVE")
print("=" * 80)
print(f" Chunks procesados: {len(df_recursive)}")
print(f" Embeddings generados: {embeddings_recursive.shape}")
print(f"  Índice FAISS: {index_recursive.ntotal} vectores")
print(f"\n Archivos generados:")
print(f"   - {index_path}")
print(f"   - {metadata_path}")
print(f"   - {backup_path}")
print("=" * 80)


PROCESANDO CHUNKS RECURSIVE
 Chunks cargados: 988 registros
   Columnas: ['fuente', 'semana', 'fecha', 'apunte', 'titulo', 'autor', 'chunk_id', 'chunk_text', 'chunk_len']
 Textos a procesar: 988

GENERANDO EMBEDDINGS
Generando embeddings para 988 textos...
   Modelo: text-embedding-3-small
   Batch size: 100
   Normalización: Sí


Generando embeddings: 100%|██████████| 10/10 [00:11<00:00,  1.16s/batch]



Embeddings generados: shape (988, 1536)
   Rango de valores: [-0.1614, 0.1802]

 Embeddings generados: (988, 1536)

CREANDO ÍNDICE FAISS
 Índice FAISS creado
   Tipo: IndexFlatIP (Inner Product / Cosine Similarity)
   Dimensión: 1536
   Vectores almacenados: 988

GUARDANDO ÍNDICE Y METADATA
 Índice guardado: vectordb\faiss_index_recursive.bin
 Metadata guardada: vectordb\faiss_index_recursive_metadata.json

GUARDANDO BACKUP CON EMBEDDINGS
 Backup guardado: data\embeddings_recursive_v1.parquet

 PROCESAMIENTO COMPLETADO: CHUNKS RECURSIVE
 Chunks procesados: 988
 Embeddings generados: (988, 1536)
  Índice FAISS: 988 vectores

 Archivos generados:
   - vectordb\faiss_index_recursive.bin
   - vectordb\faiss_index_recursive_metadata.json
   - data\embeddings_recursive_v1.parquet
